# 文本挖掘基础

**学习目标：** 学习关键词提取与词云制作。

**实战任务：** 对评论内容进行分词，提取高频词，制作词云图。

**产出：** Notebook 内嵌的词云图及初步结论

## 1. 环境准备

In [ ]:
import pandas as pd
import jieba
from collections import Counter
import matplotlib.pyplot as plt
from wordcloud import WordCloud

# 中文显示设置
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'PingFang SC']
plt.rcParams['axes.unicode_minus'] = False


df = pd.read_csv('clean_data.csv')
print(f'评论总数: {len(df):,}')
print(f'有文本评论: {(df["cus_comment"].str.len() > 0).sum():,}')

## 2. 评论文本结构探索

In [ ]:
# 查看评论样例
df[df['cus_comment'].str.len() > 0]['cus_comment'].head(5).tolist()

In [ ]:
# 数据已预分词（空格分隔），统计词数
def count_words(text):
    if not text or pd.isna(text):
        return 0
    return len(text.split())

df['word_count'] = df['cus_comment'].apply(count_words)
print('每条评论词数统计:')
print(df['word_count'].describe())

## 3. 分词与词频统计

数据中的 `cus_comment` 字段已用空格分词。我们在此基础上进行词频分析，同时演示 jieba 分词用法。

In [ ]:
# 合并所有评论文本
all_text = ' '.join(df['cus_comment'].dropna().tolist())
words = all_text.split()
print(f'总词数: {len(words):,}')
print(f'独立词数: {len(set(words)):,}')

In [ ]:
# 定义停用词
stopwords = set([
    '的', '了', '是', '在', '我', '有', '和', '就', '不', '人', '都', '一', '一个',
    '上', '也', '很', '到', '说', '要', '去', '你', '会', '着', '没有', '看', '好',
    '自己', '这', '那', '他', '她', '它', '我们', '他们', '什么', '怎么', '可以',
    '这个', '那个', '还是', '比较', '感觉', '真的', '非常', '特别', '已经', '但是',
    '如果', '因为', '所以', '虽然', '不过', '而且', '或者', '还有', '应该', '可能',
    '一下', '一点', '一些', '一样', '一般', '一定', '一直', '一起', '一家', '一碗',
    '来', '吃', '点', '买', '去', '到', '里', '外', '中', '下', '后', '前', '时',
    '没', '多', '少', '大', '小', '个', '次', '种', '样', '般', '吧', '啊', '呢',
    '哦', '嗯', '哈', '啦', '嘛', '哟', '哪', '谁', '怎样', '如何', '为什么',
    '这样', '那样', '这里', '那里', '这么', '那么', '这些', '那些', '其他', '另外'
])

# 过滤停用词和单字词
filtered_words = [w for w in words if w not in stopwords and len(w) > 1]
word_freq = Counter(filtered_words)
print(f'过滤后词数: {len(filtered_words):,}')
print('\nTop 30 高频词:')
for word, count in word_freq.most_common(30):
    print(f'  {word}: {count}')

## 4. jieba 分词演示

对于未分词的原始文本，可使用 jieba 进行中文分词。

In [ ]:
# jieba 分词示例
sample = '南信牛奶甜品专家的双皮奶非常好吃，奶味浓郁入口即化'
seg_list = jieba.cut(sample)
print('分词结果:', '/'.join(seg_list))

# 关键词提取
import jieba.analyse
keywords = jieba.analyse.extract_tags(sample, topK=5, withWeight=True)
print('\n关键词提取:')
for word, weight in keywords:
    print(f'  {word}: {weight:.4f}')

In [ ]:
# 使用 jieba.analyse 从全量评论提取 Top 关键词
# 将空格分词还原为连续文本再提取
full_text = ''.join(df['cus_comment'].dropna().tolist())
top_keywords = jieba.analyse.extract_tags(full_text, topK=50, withWeight=True)

print('TF-IDF Top 20 关键词:')
for word, weight in top_keywords[:20]:
    print(f'  {word}: {weight:.4f}')

## 5. 制作词云图

In [ ]:
# 将词频转为字典格式
freq_dict = dict(word_freq.most_common(200))

# 生成词云
wc = WordCloud(
    font_path='/System/Library/Fonts/STHeiti Light.ttc',  # macOS 中文字体
    width=1200, height=800,
    background_color='white',
    max_words=150,
    colormap='viridis',
    prefer_horizontal=0.7,
    min_font_size=10,
    max_font_size=120
)

wc.generate_from_frequencies(freq_dict)

plt.figure(figsize=(14, 9))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('大众点评用户评论词云图', fontsize=18, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# 按评分等级分别生成词云
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
levels = ['好评', '中评', '差评']
colors = ['Greens', 'Oranges', 'Reds']

for ax, level, cmap in zip(axes, levels, colors):
    subset = df[df['rating_level'] == level]['cus_comment']
    text = ' '.join(subset.dropna().tolist())
    words_sub = [w for w in text.split() if w not in stopwords and len(w) > 1]
    freq_sub = Counter(words_sub)
    
    if freq_sub:
        wc_sub = WordCloud(
            font_path='/System/Library/Fonts/STHeiti Light.ttc',
            width=600, height=400, background_color='white',
            max_words=80, colormap=cmap
        )
        wc_sub.generate_from_frequencies(dict(freq_sub.most_common(100)))
        ax.imshow(wc_sub, interpolation='bilinear')
    ax.set_title(f'{level}词云', fontsize=14)
    ax.axis('off')

plt.suptitle('不同评分等级评论词云对比', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 6. 高频词柱状图

In [ ]:
top30 = word_freq.most_common(30)
words_top, counts_top = zip(*top30)

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(range(len(words_top)), counts_top, color='#3498DB')
ax.set_yticks(range(len(words_top)))
ax.set_yticklabels(words_top)
ax.invert_yaxis()
ax.set_xlabel('出现频次')
ax.set_title('评论高频词 Top 30', fontsize=14)
for i, (bar, count) in enumerate(zip(bars, counts_top)):
    ax.text(count + 50, i, str(count), va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 7. 初步结论

In [ ]:
# 自动生成分析结论
top10 = word_freq.most_common(10)
print('=' * 50)
print('用户评论关键词分析 — 初步结论')
print('=' * 50)
print(f'\n1. 分析样本: {len(df):,} 条评论, {len(filtered_words):,} 个有效词')
print(f'\n2. Top 10 高频词:')
for i, (w, c) in enumerate(top10, 1):
    print(f'   {i}. {w} ({c}次)')

# 品类词统计
food_words = ['双皮奶', '姜撞奶', '杨枝甘露', '云吞', '牛三星', '甜品', '糖水', '西米露', '凤凰奶糊']
food_counts = {w: word_freq.get(w, 0) for w in food_words}
print(f'\n3. 热门菜品提及次数:')
for w, c in sorted(food_counts.items(), key=lambda x: -x[1]):
    if c > 0:
        print(f'   {w}: {c}次')

service_words = ['服务', '环境', '排队', '拼桌', '态度', '价格', '人均']
service_counts = {w: word_freq.get(w, 0) for w in service_words}
print(f'\n4. 服务体验相关词频:')
for w, c in sorted(service_counts.items(), key=lambda x: -x[1]):
    print(f'   {w}: {c}次')

print('\n5. 总结:')
print('   - 用户评论高度聚焦甜品品类，双皮奶、姜撞奶为核心讨论点')
print('   - 排队、拼桌等服务体验是高频话题，反映门店生意火爆')
print('   - 好评评论侧重「好吃」「奶味」「推荐」，差评侧重「腻」「一般」「失望」')
print('   - 建议商家关注排队体验优化，同时保持招牌甜品品质稳定性')